# Group member:
Ulyana Klyuchnikova, Егор Антонов, Мария, Hu Jinnan

In [32]:
import pandas as pd
import numpy as np

# Advertising Budgets of Candidates

According to the total advertising expenditure data for the 2024 U.S. presidential election from the media monitoring firm AdImpact and a summary by NBC News (2024), we set the budgets for the two major candidates as follows:

- Kamala Harris (Democrat): $880 million

- Donald Trump (Republican): $425 million

These figures include spending by the candidates' own campaigns, joint fundraising committees, and their respective national party committees.

Data source: NBC News citing AdImpact, available at:
[https://www.nbcnews.com/politics/2024-election/final-price-tag-2024-political-advertising-almost-11-billion-rcna179341](https://www.nbcnews.com/politics/2024-election/final-price-tag-2024-political-advertising-almost-11-billion-rcna179341)


In [33]:
budget_Harris = 880
budget_Trump = 425

# Electoral Vote Data

The allocation of electoral votes is based on the **2020 United States Census**, effective for the 2024 and 2028 presidential elections. Each state’s electoral votes equal its number of U.S. Representatives (determined by population) plus its two Senators. Washington, D.C., receives three electoral votes as per the 23rd Amendment. Total electors: 538; required to win: 270.

In [34]:
electoral_votes = {
    "ALABAMA": 9, "ALASKA": 3, "ARIZONA": 11, "ARKANSAS": 6, "CALIFORNIA": 54,
    "COLORADO": 10, "CONNECTICUT": 7, "DELAWARE": 3, "DISTRICT OF COLUMBIA": 3, "FLORIDA": 30,
    "GEORGIA": 16, "HAWAII": 4, "IDAHO": 4, "ILLINOIS": 19, "INDIANA": 11,
    "IOWA": 6, "KANSAS": 6, "KENTUCKY": 8, "LOUISIANA": 8, "MAINE": 4,
    "MARYLAND": 10, "MASSACHUSETTS": 11, "MICHIGAN": 15, "MINNESOTA": 10, "MISSISSIPPI": 6,
    "MISSOURI": 10, "MONTANA": 4, "NEBRASKA": 5, "NEVADA": 6, "NEW HAMPSHIRE": 4,
    "NEW JERSEY": 14, "NEW MEXICO": 5, "NEW YORK": 28, "NORTH CAROLINA": 16, "NORTH DAKOTA": 3,
    "OHIO": 17, "OKLAHOMA": 7, "OREGON": 8, "PENNSYLVANIA": 19, "RHODE ISLAND": 4,
    "SOUTH CAROLINA": 9,"SOUTH DAKOTA": 3, "TENNESSEE": 11, "TEXAS": 40, "UTAH": 6,
    "VERMONT": 3,"VIRGINIA": 13, "WASHINGTON": 12, "WEST VIRGINIA": 4, "WISCONSIN": 10, "WYOMING": 3}

# State partisan bias coefficients

Computes state-level partisan bias coefficients (γ) using MIT Election Lab data
(https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/VOQCHQ).

**Parameters**
- `dataset_url` : Path or URL to CSV file; must contain columns: year, state, party, candidatevotes
- `years` : optional, list of election years (e.g., [2020, 2024]); if None, all available years are used

**Returns**
- DataFrame with columns: `state` (state name, sorted alphabetically), `gamma` (bias coefficient)

**Processing logic**
- Keeps only rows where party is DEMOCRAT or REPUBLICAN
- When multiple years are provided, γ is the average over those years
- Smoothing: adds 0.5 to both Democratic and Republican votes before computing the logit to avoid zero-vote issues

**Interpretation of γ**
- γ > 0 : The state leans toward the Democratic candidate.
- γ ≈ 0 : The state is a swing state — no significant inherent bias.
- γ < 0 : The state leans toward the Republican candidate.
- Magnitude of |γ| : The larger the absolute value, the stronger the partisan bias.

In [35]:
def compute_state_bias_coefficients(dataset_url: str, years: list = None):
    # Place the data file and the script in the same directory
    df = pd.read_csv(dataset_url)

    # Keep only Democrat and Republican rows
    df = df[df['party'].isin(['DEMOCRAT', 'REPUBLICAN'])].copy()
    # Filter by years if provided
    if years:
        df = df[df['year'].isin(years)]

    # Aggregate by year, state, party and pivot to wide format
    pivot = df.groupby(['year', 'state', 'party'])['candidatevotes'].sum().unstack(fill_value=0)
    pivot.columns = ['dem_votes', 'rep_votes']

    # Smoothing (add 0.5 to both parties) and compute gamma = log(dem_smoothed / rep_smoothed)
    smooth = 0.5
    dem_s = pivot['dem_votes'] + smooth
    rep_s = pivot['rep_votes'] + smooth
    pivot['gamma'] = np.log(dem_s / rep_s)

    # Average gamma across years for each state, then sort alphabetically by state name
    result = pivot.groupby('state')['gamma'].mean().reset_index()
    result = result.sort_values('state').reset_index(drop=True)
    result.columns = ['state', 'gamma']
    return result

In [36]:
state_bias_coefficients = compute_state_bias_coefficients(dataset_url="countypres_2000-2024.csv", years=[2020, 2024])
print(state_bias_coefficients)

                   state     gamma
0                ALABAMA -0.583440
1                 ALASKA -0.243424
2                ARIZONA -0.065564
3               ARKANSAS -0.616609
4             CALIFORNIA  0.518602
5               COLORADO  0.253082
6            CONNECTICUT  0.355449
7               DELAWARE  0.345683
8   DISTRICT OF COLUMBIA  2.736769
9                FLORIDA -0.166908
10               GEORGIA -0.019524
11                HAWAII  0.550373
12                 IDAHO -0.723394
13              ILLINOIS  0.286763
14               INDIANA -0.360895
15                  IOWA -0.221801
16                KANSAS -0.316618
17              KENTUCKY -0.591298
18             LOUISIANA -0.422686
19                 MAINE  0.155710
20              MARYLAND  0.658888
21         MASSACHUSETTS  0.621858
22              MICHIGAN  0.013853
23             MINNESOTA  0.116381
24           MISSISSIPPI -0.405020
25              MISSOURI -0.346983
26               MONTANA -0.378311
27              NEBR

# Positive and Negative Advertising Effects
The empirical estimates for the differential effects of positive and negative advertising are derived from Gordon, B. R., Lovett, M. J., Luo, B., & Reeder, J. C. III (2022). Disentangling the Effects of Ad Tone on Voter Turnout and Candidate Choice in Presidential Elections. Management Science, 69(1), 220–243.

Using data from the 2000 and 2004 U.S. presidential elections, the study finds that positive and negative advertisements play fundamentally different roles. Negative advertising is more effective at driving relative candidate share (persuasion), whereas positive advertising stimulates voter turnout (mobilization). The estimated elasticities of the two ad tones with respect to relative candidate share are:

Positive ads: 0.016

Negative ads: 0.025

These values indicate that negative advertising has a stronger marginal impact on vote share per dollar spent than positive advertising.

Paper link: https://pubsonline.informs.org/doi/full/10.1287/mnsc.2022.4347

In [37]:
positive_effect = 0.016
negative_effect = 0.025

# Single-State Expected Electoral Votes

Uses pre‑defined global variables:
- `electoral_votes` : dictionary `state → electoral votes`
- `state_bias_coefficients` : DataFrame with columns `state` and `gamma`
- `positive_effect = 0.016` , `negative_effect = 0.025` (advertising efficiency, Gordon et al., 2022)

**Function `state_outcome(spending, state_name)`**
- `spending` : `[A_pos, A_neg, B_pos, B_neg]` – spending on positive/negative ads (units in million $)
- `state_name` : string, must exist in both `electoral_votes` and `state_bias_coefficients`
- Returns `[votes_Harris, votes_Trump]` – expected electoral votes in that state

**Model**

For state \(s\) with electoral votes \(V_s\) and bias \(\gamma_s\):

\[
P_{\text{Harris}} = \frac{\exp(\lambda^+ s_{H}^{+} + \lambda^- s_{H}^{-} + \gamma_s)}{\exp(\lambda^+ s_{H}^{+} + \lambda^- s_{H}^{-} + \gamma_s) + \exp(\lambda^+ s_{T}^{+} + \lambda^- s_{T}^{-})}
\]

\[
EV_{\text{Harris}} = P_{\text{Harris}} \times V_s ,\qquad EV_{\text{Trump}} = V_s - EV_{\text{Harris}}
\]

In [38]:
def state_outcome(spending, state_name):
    """
    spending: [A_pos, A_neg, B_pos, B_neg]  (Harris positive, Harris negative, Trump positive, Trump negative)
    state_name: string, must be a key in electoral_votes and present in state_bias_coefficients
    returns: [votes_Harris, votes_Trump]  expected electoral votes in that state
    """
    # Retrieve electoral votes for the state
    V = electoral_votes[state_name]

    # Retrieve gamma for the state from precomputed coefficients
    gamma = state_bias_coefficients.loc[state_bias_coefficients['state'] == state_name, 'gamma'].values[0]

    A_pos, A_neg, B_pos, B_neg = spending

    exp_A = np.exp(positive_effect * A_pos + negative_effect * A_neg + gamma)
    exp_B = np.exp(positive_effect * B_pos + negative_effect * B_neg)

    prob_A = exp_A / (exp_A + exp_B)

    votes_A = int(round(prob_A * V))
    votes_B = V - votes_A

    return [votes_A, votes_B]

In [39]:
state_example = state_outcome(spending=[1,2,3,4], state_name="ALABAMA")
print(state_example)

[3, 6]


# Election Outcome Function

**Function `election_outcome(spending_by_state)`**

- `spending_by_state` : list of length 51, each element is `[A_pos, A_neg, B_pos, B_neg]` for one state
  (states are processed in alphabetical order, matching `electoral_votes` keys)

- Calls `state_outcome` for each state, sums expected electoral votes, and determines the winner.

- Returns `[1, 0]` if Harris wins, `[0, 1]` if Trump wins.

In [40]:
def election_outcome(spending_by_state):
    """
    spending_by_state: list of length 51, each element is [A_pos, A_neg, B_pos, B_neg] for that state
    returns: [1, 0] if Harris wins, [0, 1] if Trump wins
    """
    # 预算检查
    total_A = 0
    total_B = 0
    for spending in spending_by_state:
        A_pos, A_neg, B_pos, B_neg = spending
        total_A += A_pos + A_neg
        total_B += B_pos + B_neg

    if total_A > budget_Harris:
        raise ValueError(f"Harris total spending ({total_A}) exceeds budget ({budget_Harris})")
    if total_B > budget_Trump:
        raise ValueError(f"Trump total spending ({total_B}) exceeds budget ({budget_Trump})")

    # 计算选举人票
    harris_votes = 0
    states = sorted(electoral_votes.keys())

    for idx, state in enumerate(states):
        votes = state_outcome(spending_by_state[idx], state)
        harris_votes += votes[0]

    trump_votes = 538 - harris_votes

    return [1, 0] if harris_votes > trump_votes else [0, 1]

In [41]:
#Election example
np.random.seed(42)
states = sorted(electoral_votes.keys())
election_example_cost = [[np.random.randint(0, 3), 0, np.random.randint(0, 3), 0] for _ in states]
election_example = election_outcome(spending_by_state=election_example_cost)
print(election_example)

[1, 0]
